## 04a_feature_engineering — RCM Intelligence Platform
### Purpose
Engineers the ML feature set for denial risk prediction.
Reads from Gold and Silver tables, creates a clean feature
matrix ready for MLflow model training.

### What this does
1. Reads fact_claims and hospital_scorecard from Gold
2. Engineers categorical and numerical features
3. Encodes categoricals with StringIndexer
4. Assembles feature vector
5. Writes ML-ready dataset to rcm_ml schema
6. Logs feature metadata to audit table

### Features engineered
### Numerical
- avg_submitted_charge, avg_total_payment, avg_medicare_payment
- charge_to_payment_ratio, revenue_gap_pct, medicare_payment_pct
- patient_responsibility, total_discharges, rcm_health_score

### Categorical (encoded)
- provider_state, hospital_type, hospital_ownership
- drg_severity_tier, rural_urban_classification

### Target variable
- denial_proxy: 1 if underpayment_flag=1 AND ctp_ratio > 75th percentile

### Outputs
- rcm_platform.rcm_ml.features_denial_risk

| Field      | Value |
|------------|-------|
| Author     | Mayank Joshi |
| Layer      | ML |
| Run order  | 8 — after 03b_kpi_aggregations |
| Depends on | 00_config, 00_utils |

In [0]:
%run /Users/jmayank574@gmail.com/rcm-intelligence-platform/00_setup/00_config

In [0]:
%run /Users/jmayank574@gmail.com/rcm-intelligence-platform/00_setup/00_utils

In [0]:
# ============================================================
# BATCH METADATA + ML TABLE
# ============================================================

import uuid
from datetime import datetime, timezone
from pyspark.sql.functions import (
    col, lit, when, expr,
    percentile_approx, avg, stddev
)

BATCH_ID        = str(uuid.uuid4())
BATCH_DATE      = datetime.now(timezone.utc).strftime("%Y-%m-%d")
BATCH_TIMESTAMP = datetime.now(timezone.utc).strftime("%Y-%m-%d %H:%M:%S")
NOTEBOOK_NAME   = "04a_feature_engineering"

# ML feature table
TBL_ML_FEATURES = f"{ML_DB}.features_denial_risk"

print(f"Batch ID        : {BATCH_ID}")
print(f"Batch date      : {BATCH_DATE}")
print(f"ML features tbl : {TBL_ML_FEATURES}")

In [0]:
# ============================================================
# STEP 1 — READ SOURCE TABLES (including IPPS)
# ============================================================

print("=" * 55)
print("  READING SOURCE TABLES")
print("=" * 55)

df_fact      = spark.table(TBL_FACT_CLAIMS)
df_scorecard = spark.table(TBL_GOLD_SCORECARD)
df_ipps      = spark.table("rcm_platform.rcm_silver.ipps_payment_impacts")

print(f"fact_claims       : {df_fact.count():,} rows")
print(f"hospital_scorecard: {df_scorecard.count():,} rows")
print(f"ipps_payment_impacts: {df_ipps.count():,} rows")

# Get the 75th percentile of CTP ratio
# Used to define the denial proxy target variable
ctp_75th = df_fact.select(
    percentile_approx("charge_to_payment_ratio", 0.75)
    .alias("p75")
).collect()[0]["p75"]

# Force to Python float — prevents Spark type issues
ctp_75th = float(ctp_75th)

print(f"\nCTP ratio 75th percentile : {ctp_75th:.4f}")
print("Claims above this + underpayment flag = denial proxy = 1")

In [0]:
# ============================================================
# STEP 2 — BUILD FEATURE MATRIX (with IPPS features)
# Added 3 new engineered features + 4 IPPS payment features
# ============================================================

print("=" * 55)
print("  BUILDING FEATURE MATRIX (enhanced + IPPS)")
print("=" * 55)

# Get hospital-level RCM score from scorecard
df_hosp_score = df_scorecard.select(
    col("provider_id"),
    col("rcm_health_score"),
    col("rcm_health_grade"),
    col("avg_ctp_ratio").alias("hospital_avg_ctp"),
    col("underpayment_rate_pct").alias("hospital_underpayment_rate"),
    col("total_discharges").alias("hospital_total_discharges"),
    col("drg_count").alias("hospital_drg_count")
)

# Get DRG-level national benchmarks from Gold KPI table
df_drg_benchmarks = spark.table(TBL_GOLD_KPI_DRG).select(
    col("drg_code"),
    col("avg_charge").alias("drg_national_avg_charge"),
    col("avg_ctp_ratio").alias("drg_national_avg_ctp"),
    col("underpayment_rate_pct").alias("drg_national_underpayment_rate")
)

# Get IPPS payment features (join on provider_number = provider_id)
df_ipps_features = df_ipps.select(
    col("provider_number").alias("provider_id"),
    col("fy2024_operating_case_mix_index").alias("ipps_case_mix_index"),
    col("dsh_percentage").alias("ipps_dsh_pct"),
    col("resident_to_bed_ratio").alias("ipps_resident_to_bed_ratio"),
    col("operating_payment_change_pct").alias("ipps_payment_change_pct")
)

# Calculate medians for NULL imputation (for 8.2% hospitals without IPPS match)
print("\nCalculating medians for IPPS feature imputation...")
medians = df_ipps_features.select(
    percentile_approx("ipps_case_mix_index", 0.5).alias("median_cmi"),
    percentile_approx("ipps_dsh_pct", 0.5).alias("median_dsh"),
    percentile_approx("ipps_resident_to_bed_ratio", 0.5).alias("median_rtb"),
    percentile_approx("ipps_payment_change_pct", 0.5).alias("median_pct")
).collect()[0]

median_cmi = float(medians["median_cmi"])
median_dsh = float(medians["median_dsh"])
median_rtb = float(medians["median_rtb"])
median_pct = float(medians["median_pct"])

print(f"  Median case_mix_index         : {median_cmi:.4f}")
print(f"  Median dsh_pct                : {median_dsh:.4f}")
print(f"  Median resident_to_bed_ratio  : {median_rtb:.4f}")
print(f"  Median payment_change_pct     : {median_pct:.2f}%")

# Join everything
df_features = df_fact \
    .join(df_hosp_score,      on="provider_id", how="left") \
    .join(df_drg_benchmarks,  on="drg_code",    how="left") \
    .join(df_ipps_features,   on="provider_id", how="left") \
    .select(
        col("provider_id"),
        col("drg_code"),
        col("provider_state"),

        # Pre-submission numerical features
        col("avg_submitted_charge"),
        col("total_discharges"),
        col("rcm_health_score"),
        col("hospital_avg_ctp"),
        col("hospital_underpayment_rate"),
        col("hospital_total_discharges"),
        col("hospital_drg_count"),
        col("high_volume_flag"),

        # Engineered Feature 1: DRG national underpayment rate
        col("drg_national_underpayment_rate"),

        # Engineered Feature 2: charge vs DRG national benchmark
        when(col("drg_national_avg_charge") > 0,
            col("avg_submitted_charge") / col("drg_national_avg_charge")
        ).otherwise(lit(1.0)).alias("charge_vs_drg_benchmark"),

        # Engineered Feature 3: provider DRG specialisation
        when(col("hospital_total_discharges") > 0,
            col("total_discharges") / col("hospital_total_discharges")
        ).otherwise(lit(0.0)).alias("drg_specialisation_ratio"),

        # IPPS Feature 1: Case Mix Index (patient complexity)
        coalesce(col("ipps_case_mix_index"), lit(median_cmi)).alias("case_mix_index"),

        # IPPS Feature 2: DSH % (safety net hospital proxy)
        coalesce(col("ipps_dsh_pct"), lit(median_dsh)).alias("dsh_pct"),

        # IPPS Feature 3: Resident-to-bed ratio (teaching intensity)
        coalesce(col("ipps_resident_to_bed_ratio"), lit(median_rtb)).alias("resident_to_bed_ratio"),

        # IPPS Feature 4: Payment change % (payer tightening signal)
        coalesce(col("ipps_payment_change_pct"), lit(median_pct)).alias("operating_payment_change_pct"),

        # Categoricals
        col("hospital_type"),
        col("hospital_ownership"),
        col("drg_severity_tier"),
        col("rural_urban_classification"),
        col("emergency_services"),

        # Label
        col("underpayment_flag").alias("denial_proxy"),
        col("underpayment_flag")
    ) \
    .withColumn("_feature_batch_id", lit(BATCH_ID)) \
    .withColumn("_feature_date",     lit(BATCH_DATE)) \
    .dropna(subset=[
        "avg_submitted_charge",
        "rcm_health_score",
        "hospital_avg_ctp"
    ])

feature_count = df_features.count()
denial_count  = df_features.filter("denial_proxy = 1").count()
denial_rate   = float(denial_count) / float(feature_count) * 100

print(f"\nTotal rows       : {feature_count:,}")
print(f"Denial proxy = 1 : {denial_count:,} ({denial_rate:.2f}%)")
print(f"Denial proxy = 0 : {feature_count - denial_count:,} ({100-denial_rate:.2f}%)")
print(f"\nEngineered features (3):")
print(f"  drg_national_underpayment_rate — DRG underpayment frequency nationally")
print(f"  charge_vs_drg_benchmark        — hospital charge vs national DRG average")
print(f"  drg_specialisation_ratio       — DRG volume concentration at hospital")
print(f"\nIPPS payment features (4):")
print(f"  case_mix_index                 — patient complexity (coding risk)")
print(f"  dsh_pct                        — safety net payer mix proxy")
print(f"  resident_to_bed_ratio          — teaching intensity (coding patterns)")
print(f"  operating_payment_change_pct   — YoY Medicare rate pressure")

In [0]:
# ============================================================
# STEP 3 — ENCODE CATEGORICALS + WRITE FEATURE TABLE
# ============================================================

print("=" * 55)
print("  ENCODING CATEGORICALS + WRITING FEATURES")
print("=" * 55)

# Fill nulls
df_filled = df_features \
    .fillna("Unknown", subset=[
        "hospital_type",
        "hospital_ownership",
        "drg_severity_tier",
        "rural_urban_classification",
        "emergency_services"
    ]) \
    .fillna(0.0, subset=[
        "rcm_health_score",
        "hospital_avg_ctp",
        "hospital_underpayment_rate",
        "hospital_total_discharges",
        "hospital_drg_count",
        "drg_national_underpayment_rate",
        "charge_vs_drg_benchmark",
        "drg_specialisation_ratio",
        "case_mix_index",
        "dsh_pct",
        "resident_to_bed_ratio",
        "operating_payment_change_pct"
    ])

df_filled.createOrReplaceTempView("features_raw")

df_encoded = spark.sql("""
    SELECT
        provider_id,
        drg_code,
        provider_state,

        -- Original numerical features
        avg_submitted_charge,
        total_discharges,
        rcm_health_score,
        hospital_avg_ctp,
        hospital_underpayment_rate,
        hospital_total_discharges,
        hospital_drg_count,
        high_volume_flag,

        -- Engineered features
        drg_national_underpayment_rate,
        charge_vs_drg_benchmark,
        drg_specialisation_ratio,

        -- IPPS payment features
        case_mix_index,
        dsh_pct,
        resident_to_bed_ratio,
        operating_payment_change_pct,

        -- Categorical raw
        hospital_type,
        hospital_ownership,
        drg_severity_tier,
        rural_urban_classification,
        emergency_services,

        -- Categorical encoded
        DENSE_RANK() OVER (ORDER BY hospital_type)              AS hospital_type_idx,
        DENSE_RANK() OVER (ORDER BY hospital_ownership)         AS hospital_ownership_idx,
        DENSE_RANK() OVER (ORDER BY drg_severity_tier)          AS drg_severity_tier_idx,
        DENSE_RANK() OVER (ORDER BY rural_urban_classification)  AS rural_urban_classification_idx,
        DENSE_RANK() OVER (ORDER BY provider_state)             AS provider_state_idx,

        -- Target and flags
        underpayment_flag,
        denial_proxy,
        _feature_batch_id,
        _feature_date
    FROM features_raw
""")

# Updated feature list — 20 features (11 original + 3 engineered + 4 IPPS + 5 categorical)
numerical_features = [
    "avg_submitted_charge",
    "total_discharges",
    "rcm_health_score",
    "hospital_avg_ctp",
    "hospital_underpayment_rate",
    "hospital_total_discharges",
    "hospital_drg_count",
    "high_volume_flag",
    "drg_national_underpayment_rate",
    "charge_vs_drg_benchmark",
    "drg_specialisation_ratio",
    "case_mix_index",
    "dsh_pct",
    "resident_to_bed_ratio",
    "operating_payment_change_pct"
]

categorical_features_idx = [
    "hospital_type_idx",
    "hospital_ownership_idx",
    "drg_severity_tier_idx",
    "rural_urban_classification_idx",
    "provider_state_idx"
]

all_features = numerical_features + categorical_features_idx

print(f"Feature vector size : {len(all_features)} features")
print(f"Numerical features  : {len(numerical_features)}")
print(f"Categorical features: {len(categorical_features_idx)}")
print(f"New features added  : 7 (3 engineered + 4 IPPS)")

df_encoded.select(
    "provider_id",
    "drg_code",
    "provider_state",
    *numerical_features,
    *categorical_features_idx,
    "hospital_type",
    "hospital_ownership",
    "drg_severity_tier",
    "rural_urban_classification",
    "emergency_services",
    "underpayment_flag",
    "denial_proxy",
    "_feature_batch_id",
    "_feature_date"
).write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable(TBL_ML_FEATURES)

ml_rows = spark.table(TBL_ML_FEATURES).count()
print(f"\nML feature table : {TBL_ML_FEATURES}")
print(f"Rows written     : {ml_rows:,}")

display(
    spark.table(TBL_ML_FEATURES).select(
        "provider_id",
        "drg_code",
        "hospital_underpayment_rate",
        "drg_national_underpayment_rate",
        "case_mix_index",
        "dsh_pct",
        "resident_to_bed_ratio",
        "operating_payment_change_pct",
        "denial_proxy"
    ).limit(10)
)

In [0]:
# ============================================================
# STEP 4 — FEATURE STATISTICS
# ============================================================

print("=" * 55)
print("  FEATURE STATISTICS")
print("=" * 55)

spark.sql(f"""
    SELECT
        ROUND(AVG(avg_submitted_charge), 2)          AS avg_submitted_charge,
        ROUND(AVG(total_discharges), 0)              AS avg_discharges,
        ROUND(AVG(rcm_health_score), 1)              AS avg_rcm_score,
        ROUND(AVG(hospital_avg_ctp), 2)              AS avg_hospital_ctp,
        ROUND(AVG(hospital_underpayment_rate), 2)    AS avg_hospital_underpayment_rate,
        ROUND(AVG(drg_national_underpayment_rate), 2) AS avg_drg_national_underpayment_rate,
        ROUND(AVG(charge_vs_drg_benchmark), 4)       AS avg_charge_vs_benchmark,
        ROUND(AVG(drg_specialisation_ratio), 4)      AS avg_drg_specialisation,
        SUM(denial_proxy)                            AS total_denial_proxy_1,
        COUNT(*)                                     AS total_rows,
        ROUND(SUM(denial_proxy)/COUNT(*)*100, 2)     AS denial_proxy_rate_pct
    FROM {TBL_ML_FEATURES}
""").show(truncate=False)

print("Denial proxy by DRG severity tier:")
spark.sql(f"""
    SELECT
        drg_severity_tier,
        COUNT(*)                                  AS rows,
        SUM(denial_proxy)                         AS denial_count,
        ROUND(SUM(denial_proxy)/COUNT(*)*100, 2)  AS denial_rate_pct,
        ROUND(AVG(drg_national_underpayment_rate), 2) AS avg_drg_national_underpayment
    FROM {TBL_ML_FEATURES}
    GROUP BY drg_severity_tier
    ORDER BY denial_rate_pct DESC
""").show(truncate=False)

print("Denial proxy by rural/urban classification:")
spark.sql(f"""
    SELECT
        rural_urban_classification,
        COUNT(*)                                  AS rows,
        SUM(denial_proxy)                         AS denial_count,
        ROUND(SUM(denial_proxy)/COUNT(*)*100, 2)  AS denial_rate_pct,
        ROUND(AVG(charge_vs_drg_benchmark), 3)    AS avg_charge_vs_benchmark
    FROM {TBL_ML_FEATURES}
    GROUP BY rural_urban_classification
    ORDER BY denial_rate_pct DESC
""").show(truncate=False)

print("New feature distributions:")
spark.sql(f"""
    SELECT
        ROUND(MIN(charge_vs_drg_benchmark), 3)   AS min_charge_vs_benchmark,
        ROUND(AVG(charge_vs_drg_benchmark), 3)   AS avg_charge_vs_benchmark,
        ROUND(MAX(charge_vs_drg_benchmark), 3)   AS max_charge_vs_benchmark,
        ROUND(MIN(drg_specialisation_ratio), 4)  AS min_drg_specialisation,
        ROUND(AVG(drg_specialisation_ratio), 4)  AS avg_drg_specialisation,
        ROUND(MAX(drg_specialisation_ratio), 4)  AS max_drg_specialisation
    FROM {TBL_ML_FEATURES}
""").show(truncate=False)

In [0]:
# ============================================================
# STEP 5 — LOG TO AUDIT TABLE
# ============================================================

log_pipeline_run(
    notebook_name    = NOTEBOOK_NAME,
    layer            = "ml",
    status           = "SUCCESS",
    rows_read        = feature_count,
    rows_written     = ml_rows,
    rows_quarantined = 0,
    message          = (
        f"Batch {BATCH_ID} — "
        f"features: {ml_rows:,} rows | "
        f"{len(all_features)} features engineered | "
        f"denial proxy rate: {denial_rate:.2f}% | "
        f"new features: drg_national_underpayment_rate, "
        f"charge_vs_drg_benchmark, drg_specialisation_ratio, "
        f"case_mix_index, dsh_pct, resident_to_bed_ratio, operating_payment_change_pct (IPPS)"
    )
)